# Fraud Risk Analytics: UAE E-Commerce Transactions (100K)
**Task issued by:** Senior Data Science Engineer, Fraud & Risk Analytics, Amazon (MENA marketplace)
**Assigned to:** Data Scientist
**Dataset:** `uae_ecom_fraud_100k` (one row per transaction, 36 columns)
**Time budget:** 6 hours
**Deliverables:** this notebook, executed top to bottom with zero errors, plus a one-page memo (Part H) inside the notebook

---
## 1. Business context
Card-not-present fraud on our UAE marketplace is costing us chargebacks, lost inventory and customer trust. The Payments Risk team currently blocks orders with a hand-written **rule engine** (the six `fraud_flag_*` columns). The VP of Risk wants to know three things before next quarter's planning:

1. **Where is fraud concentrated?** (payment method, device, category, city, hour, account age)
2. **How good is the current rule engine?** Which rules earn their keep, and which only create false positives?
3. **Can a model beat the rules, and at what business cost threshold should we operate?**

Your analysis will be read by a Risk Director and challenged by an ML engineer. Both will check your numbers.

## 2. Data dictionary (as supplied)
| Group | Columns |
|---|---|
| Identifiers and time | `transaction_id`, `user_id`, `timestamp_utc`, `local_hour`, `odd_hour` |
| Order | `amount_aed`, `currency`, `items_count`, `avg_item_price`, `merchant_category` |
| Channel | `payment_method`, `device_type`, `browser`, `ip_address`, `ip_risk_score`, `card_present` |
| Geography | `shipping_city`, `billing_city`, `shipping_billing_match`, `bin_country`, `card_country_match` |
| Card and user | `card_age_days`, `email_domain`, `user_prev_chargebacks`, `user_is_high_risk`, `user_account_age_days` |
| Velocity | `transactions_last_24h`, `transactions_last_1h` |
| Rule-engine flags | `fraud_flag_ip`, `fraud_flag_mismatch`, `fraud_flag_velocity`, `fraud_flag_new_account`, `fraud_flag_prev_cb`, `fraud_flag_odd_hour` |
| Target | `is_fraud` (1 = confirmed fraud) |
| Lineage | `data_source` |

## 3. Ground rules
1. **Do not trust the data. Verify it.** Several columns are *derived* from others. Whether they agree is for you to test, not assume.
2. **Never type a number by hand.** Every figure in markdown, charts and the memo must be printed by code first.
3. **No hardcoded expected values in `assert` statements.** Read expected values from the data. Every assert needs a message.
4. **Think about leakage before you model.** A column that is only known *after* the outcome, or that is a rewrite of the label, is not a feature.
5. **Respect time.** Fraud data drifts. Your validation split must be chronological.
6. **Accuracy is not a metric here.** Fraud is rare. Use precision, recall, PR-AUC and money.
7. **If something cannot be computed from the data, say so.** Do not invent columns, results or business numbers. Any assumption (for example the cost of a manual review) goes in the *Assumptions* cell and is printed.
8. Keep a clean notebook: sections match Parts A to H, no duplicate load cells, no failed cells, no debug clutter.

## 4. Table of contents
- **Part A** Load, validate, audit (data quality log)
- **Part B** Exploratory fraud analysis
- **Part C** Rule-engine performance
- **Part D** Leakage review and modelling
- **Part E** Cost-based threshold and decision policy
- **Part F** SQL cross-check (SQLite)
- **Part G** Segment fairness and drift
- **Part H** Manager memo and self-review

In [1]:
# ---- Setup -----------------------------------------------------------------
import os, sqlite3, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

RANDOM_STATE = 42
DATA_PATH = "uae_ecom_fraud_100k.csv"      # put the file next to this notebook (csv or parquet)
if not os.path.exists(DATA_PATH) and os.path.exists("uae_ecom_fraud_100k.parquet"):
    DATA_PATH = "uae_ecom_fraud_100k.parquet"

EXPECTED_COLUMNS = [
    "transaction_id","user_id","timestamp_utc","amount_aed","currency","payment_method","device_type","browser",
    "merchant_category","items_count","avg_item_price","shipping_city","billing_city","shipping_billing_match",
    "ip_address","ip_risk_score","card_present","bin_country","card_age_days","card_country_match","email_domain",
    "user_prev_chargebacks","user_is_high_risk","user_account_age_days","transactions_last_24h","transactions_last_1h",
    "local_hour","odd_hour","is_fraud","fraud_flag_ip","fraud_flag_mismatch","fraud_flag_velocity",
    "fraud_flag_new_account","fraud_flag_prev_cb","fraud_flag_odd_hour","data_source",
]
FLAG_COLS = [c for c in EXPECTED_COLUMNS if c.startswith("fraud_flag_")]
TARGET = "is_fraud"

# ---- Self-review recorder: call record() at the end of every Part ------------
SELF_REVIEW = []
def record(part, requirement, passed, evidence):
    """Log a PASS/FAIL line with the printed evidence that proves it."""
    SELF_REVIEW.append({"part": part, "requirement": requirement,
                        "result": "PASS" if passed else "FAIL", "evidence": str(evidence)})

def wilson_ci(k, n, z=1.96):
    """95% Wilson confidence interval for a rate k/n (use it for every fraud-rate table)."""
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    d = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / d
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / d
    return (centre - half, centre + half)
print("Setup complete. DATA_PATH =", DATA_PATH)

Setup complete. DATA_PATH = uae_ecom_fraud_100k.csv


---
## Part A. Load, validate, audit

### Requirements
1. Load the file (CSV or parquet). Print `shape`, `dtypes`, null count per column and memory usage.
2. **Schema check:** assert that every column in `EXPECTED_COLUMNS` is present and report any extra or missing columns.
3. Parse `timestamp_utc` to timezone-aware datetime. Print min and max. State whether the data covers whole months.
4. Run the audit below and write one row per issue into an **audit log DataFrame** (`issue_id`, `table_check`, `rows_affected`, `pct_of_rows`, `decision`, `justification`). Log only issues you actually found, with real counts.
5. Apply your decisions. **Do not delete rows without a written reason.** Flag rather than drop where possible. Quarantine anything removed into `df_quarantine` with a `reason` column and show a row bridge: `raw rows - quarantined rows = clean rows`.

### Audit checklist (test every line, log what you find)
| # | Check | Hint |
|---|---|---|
| 1 | Duplicate `transaction_id`, and fully duplicated rows | Which one do you keep, and why? |
| 2 | Null counts and sentinel values (`-1`, `0`, `"unknown"`, empty string) | Do not assume nulls are the only missing marker. |
| 3 | `currency`: is every row AED? | If not, is `amount_aed` really in AED? Say what you did. |
| 4 | Negative, zero or extreme `amount_aed`, `items_count`, `avg_item_price` | Use IQR or percentile fences. Do not cap blindly. |
| 5 | `amount_aed` vs `items_count * avg_item_price` | How many rows disagree, and by how much? Shipping or fees could explain small gaps. |
| 6 | `local_hour` vs hour of `timestamp_utc` | What is the implied UTC offset? Is it constant (UAE has no daylight saving)? |
| 7 | `odd_hour` vs `local_hour` | Reverse-engineer the exact rule that defines `odd_hour`. |
| 8 | `shipping_billing_match` vs `shipping_city == billing_city` | Consider case, spacing and spelling variants of city names. |
| 9 | `card_country_match` vs `bin_country` | Which country is the comparison against? |
| 10 | `transactions_last_1h` must be <= `transactions_last_24h` | Count violations. |
| 11 | `ip_address` format and `ip_risk_score` range | Valid IPv4? Is the score bounded? |
| 12 | `card_present` vs `device_type` / `payment_method` | Is "card present" plausible for web orders? |
| 13 | `user_account_age_days`, `card_age_days` negative or implausible | |
| 14 | Text categories: case, spaces, near-duplicates in `payment_method`, `browser`, `merchant_category`, `email_domain` | |
| 15 | `data_source`: list values and row counts | Are the sources comparable? Check fraud rate per source before pooling. |
| 16 | Per user: does `user_prev_chargebacks` or `user_is_high_risk` change between a user's transactions over time? | Static attributes should not move backwards in time. |

### Acceptance checks
- Row bridge balances. Asserts read values from the data.
- Audit log contains only issues with a non-zero count, or an explicit "checked, none found" line.

In [2]:
# ---- A1. Load and first look -------------------------------------------------
# TODO: load DATA_PATH into `raw` (pd.read_csv or pd.read_parquet); keep `raw` untouched.
# TODO: print shape, dtypes, null counts, memory usage.
# TODO: assert the schema matches EXPECTED_COLUMNS (message must name any missing column).
raw = None

In [3]:
# ---- A2. Parse time and run the audit checks (one small block per check) ----------
# TODO: parse timestamp_utc with utc=True; print min, max and the number of distinct months.
# TODO: for each audit check 1-16, print the evidence and append a row to `audit_rows`.
audit_rows = []

In [4]:
# ---- A3. Apply decisions, quarantine, row bridge --------------------------------
# TODO: build `df` (clean) and `df_quarantine` (removed rows with reason).
# TODO: print the bridge and assert len(raw) - len(df_quarantine) == len(df).
# TODO: show the audit log as a DataFrame sorted by rows_affected.
df = None
df_quarantine = None
audit_log = pd.DataFrame(audit_rows)

In [5]:
# record("A", "Row bridge balances", <bool from your code>, <printed evidence>)

---
## Part B. Exploratory fraud analysis

### Requirements (print every result from code)
1. **Headline:** total transactions, fraud count, **fraud rate %**, total `amount_aed`, **fraud value (AED)** and fraud value as % of total. State clearly whether the fraud rate by *count* and by *value* differ and what that means.
2. **Fraud rate with 95% Wilson interval** by: `payment_method`, `device_type`, `browser`, `merchant_category`, `shipping_city`, `bin_country`, `email_domain` (top 15 by volume), `data_source`. Show transactions, frauds, rate and interval. Mark groups with fewer than 100 rows as *low volume* and do not draw conclusions from them.
3. **Time:** fraud rate by `local_hour` (24 points), day of week, and month. Is the "odd hour" idea supported by the data, and from which hour to which hour?
4. **Amount:** compare `amount_aed` distribution for fraud vs non-fraud (median, p90, p99). Fraud rate by amount decile.
5. **Account and card risk:** fraud rate by account-age band, card-age band, `user_prev_chargebacks` (0, 1, 2+), `transactions_last_1h` (0, 1, 2, 3+) and `ip_risk_score` band.
6. **Two-way view:** one heatmap of fraud rate by `payment_method` x `device_type`, with the transaction count shown so nobody over-reads a thin cell.
7. **Reconciliation:** every breakdown in item 2 must sum back to the total transactions and total frauds. Assert this with a tolerance of zero for counts.

### Charts (minimum 4, matplotlib or seaborn)
Fraud rate by hour with interval band; fraud rate by payment method (sorted bars); fraud rate by account-age band; amount distribution fraud vs non-fraud (log x-axis). Titles, axis labels, AED formatted like `AED 1.2M`, sorted bars, maximum 4 colours.

### Acceptance checks
- Counts reconcile. Low-volume groups are marked. Every insight is backed by a printed table.
- Write **at least 3 insights**, each in the form: *what the data shows, why it likely happens, what the business should do*. Correlation is not cause, so say so where it applies.

In [6]:
# ---- B1. Headline numbers -----------------------------------------------------------
# TODO

In [7]:
# ---- B2. Fraud rate by segment with Wilson interval (write one reusable function) ------
# TODO: def fraud_by(col, min_rows=100): ... returns transactions, frauds, rate, ci_low, ci_high, low_volume

In [8]:
# ---- B3. Time patterns, B4. Amount, B5. Account/card risk bands ---------------------------
# TODO

In [9]:
# ---- B6. Heatmap, B7. Reconciliation asserts, charts ---------------------------------------
# TODO

In [10]:
# record("B", "Breakdowns reconcile to total transactions and frauds", <bool>, <evidence>)

---
## Part C. Rule-engine performance

The six `fraud_flag_*` columns are the current production rules. Treat each as a **classifier** and judge it like one.

### Requirements
1. For each flag and for the combinations below, compute: rows flagged, **precision**, **recall**, **lift** over the base fraud rate, **false-positive rate** and **value caught** (AED of fraud caught) versus **value of good orders wrongly flagged**.
   - each single flag
   - `any flag`, `at least 2 flags`, `at least 3 flags`
2. **Rule overlap:** a table (or heatmap) of how often each pair of flags fires together. Which rules are redundant?
3. **Rule consistency (important):** each flag is supposed to be computed from other columns in the data. Try to **re-derive each flag** from its source columns (for example, `fraud_flag_odd_hour` from `local_hour`, `fraud_flag_mismatch` from `shipping_billing_match`, `fraud_flag_prev_cb` from `user_prev_chargebacks`, `fraud_flag_new_account` from `user_account_age_days`, `fraud_flag_velocity` from `transactions_last_1h` / `transactions_last_24h`, `fraud_flag_ip` from `ip_risk_score`).
   - Find the threshold that reproduces each flag as closely as possible and report the **agreement %**.
   - List any flag that cannot be reproduced and say what that tells you.
4. **How does `is_fraud` relate to the flags?** Test whether `is_fraud` is (nearly) a deterministic function of the flags, for example by fitting a very small decision tree on the six flags only and reporting its accuracy and PR-AUC. State what this means for Part D. Do not skip this step.
5. **Marginal value of each rule:** drop one flag at a time from the `any flag` rule and show the change in recall and in flagged volume.

### Acceptance checks
- Precision and recall are computed from a confusion matrix you print, not from a library call alone.
- Written answer: *which rules should be kept, tuned or retired, and why*, based on printed numbers.

In [11]:
# ---- C1. Single-flag and combined-flag metrics ----------------------------------------------
# TODO: def rule_metrics(mask, y, amount) -> dict(flagged, tp, fp, fn, tn, precision, recall, lift, fpr, value_caught, value_fp)

In [12]:
# ---- C2. Overlap, C3. Re-derive each flag, C4. Is is_fraud a function of the flags? --------------
# TODO

In [13]:
# ---- C5. Leave-one-rule-out ------------------------------------------------------------------
# TODO

In [14]:
# record("C", "Each flag re-derived with agreement % reported", <bool>, <evidence>)

---
## Part D. Leakage review and modelling

### D1. Leakage review (write this BEFORE modelling)
Create a feature table with columns: `feature`, `known_at_decision_time (yes/no)`, `leakage_risk (none/low/high)`, `reason`.
Think about each of these specifically:
- The six `fraud_flag_*` columns: are they inputs to the decision or a re-encoding of it? What did Part C step 4 show?
- `user_prev_chargebacks`, `user_is_high_risk`: when were they computed, and could they include the outcome of **this** transaction?
- `transactions_last_24h`, `transactions_last_1h`: do they include the current transaction?
- `transaction_id`, `user_id`, `ip_address`: identifiers or usable features? What is the risk of memorising users?
- `data_source`: a model that learns the source instead of fraud will fail in production. Check first.

### D2. Split
- **Chronological split:** train on the earliest ~70%, validate on the next ~15%, **test on the latest ~15%** (by `timestamp_utc`). Print the date range and fraud rate of each part.
- Users appear in several rows. State how you handled that (for example check how many test users also appear in train) and what risk remains.

### D3. Models (train on the train set only; tune on validation; touch the test set once)
Build **two feature sets** and compare them:
- **Set 1, "no rule flags":** raw behavioural and profile features only.
- **Set 2, "with rule flags":** Set 1 plus the six `fraud_flag_*` columns, only if your leakage review allows it. If you exclude them, show the result anyway so the team can see the gap.

For each feature set train:
1. **Logistic Regression** with scaling and one-hot encoding in a `Pipeline` (so there is no preprocessing leakage), class weights chosen by you.
2. **Gradient boosting** (`HistGradientBoostingClassifier` or LightGBM/XGBoost if installed).

Engineer at least 5 sensible features (examples: amount per item vs user history, log amount, hour cyclical encoding, `transactions_last_1h / (transactions_last_24h + 1)`, card age to account age ratio, high-risk email domain indicator). Justify each in one line.

### D4. Evaluation (test set only, for the final chosen setup)
- **PR-AUC (average precision)** as the headline metric, ROC-AUC as secondary.
- Precision, recall, F1 at your chosen threshold, and the **confusion matrix** (counts).
- **Recall at fixed precision** (for example, recall when precision is at least 90%) and **recall at 1% false-positive rate**. Compare against the rule engine from Part C at the same operating point.
- **Calibration:** reliability curve and Brier score. Can the probabilities be used as risk scores?
- **Feature importance:** permutation importance on the validation set. Compare it with your leakage review. If a feature you marked "high risk" tops the chart, explain it.
- A comparison table: rule engine vs each model, same test set, same metrics.

### Acceptance checks
- No fitting of any scaler, encoder or threshold on test data.
- Test set used once. Metrics table shows both feature sets.
- Written answer: *which model and feature set would you ship, and what would you tell the team about the gap between Set 1 and Set 2?*

In [15]:
# ---- D1. Leakage review table ----------------------------------------------------------------------
# TODO: build `leakage_table` (one row per candidate feature) and display it.
leakage_table = None

In [16]:
# ---- D2. Chronological split ---------------------------------------------------------------------------
# TODO: sort by timestamp_utc; build train/valid/test; print date range, rows and fraud rate for each.

In [18]:
# ---- D3. Feature engineering and pipelines ------------------------------------------------------------
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (average_precision_score, roc_auc_score, precision_recall_curve,
                             confusion_matrix, precision_score, recall_score, f1_score, brier_score_loss)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
# TODO: engineer features, define numeric and categorical column lists for Set 1 and Set 2.

In [19]:
# ---- D3b. Train and tune on train/valid ----------------------------------------------------------------
# TODO

In [20]:
# ---- D4. One-time test evaluation, comparison table, calibration, importance ----------------------------
# TODO

In [21]:
# record("D", "Chronological split, no test-set fitting, both feature sets compared", <bool>, <evidence>)

---
## Part E. Cost-based threshold and decision policy

Accuracy and AUC do not pay anyone's salary. Money does.

### Assumptions (put these in one cell, print them, and keep them adjustable)
- Cost of a **missed fraud** = the order value `amount_aed` (inventory and cash lost) plus a **chargeback fee** you set (for example `CHARGEBACK_FEE_AED`).
- Cost of a **manual review** = `REVIEW_COST_AED` per case (analyst time).
- Cost of **declining a good order** = lost margin on that order. Because the file has **no profit column**, use `MARGIN_RATE * amount_aed` and state the rate you chose.
- Treat all three numbers as assumptions to be confirmed by Finance. Do not present them as facts.

### Requirements
1. Compute **total expected cost on the test set** for: approve everything, the current rule engine (block on `any flag`), and your model at every threshold from 0.01 to 0.99.
2. Plot **cost vs threshold**. Mark the cost-minimising threshold and the one that gives 90% precision.
3. Design a **three-zone policy**: auto-approve (score below `t_low`), manual review (between), auto-decline (above `t_high`). Choose `t_low` and `t_high`, report the volumes in each zone, review workload per day, fraud caught, good orders declined and total cost.
4. **Sensitivity:** repeat the optimum with review cost and chargeback fee changed by -50% and +50%. Does the best threshold move a lot? Show a small table.
5. Report **net saving versus the current rule engine** in AED and in %, on the test set only.

### Acceptance checks
- All costs computed from code with the assumptions printed above them.
- A sentence saying how sure you are and what would change the answer.

In [22]:
# ---- E0. Assumptions (edit, then print) -------------------------------------------------------------------
CHARGEBACK_FEE_AED = None      # TODO: choose and justify
REVIEW_COST_AED    = None      # TODO: choose and justify
MARGIN_RATE        = None      # TODO: choose and justify (no profit column exists)

In [23]:
# ---- E1-E5. Cost curve, three-zone policy, sensitivity, saving vs rule engine --------------------------------
# TODO

In [24]:
# record("E", "Costs computed from printed assumptions; saving vs rules reported", <bool>, <evidence>)

---
## Part F. SQL cross-check (SQLite)

### Requirements
1. Write the cleaned DataFrame to a SQLite database (`fraud_clean.db`) as table `transactions` with **snake_case columns and ISO timestamps**. Use `if_exists="replace"` and assert `SQL row count == DataFrame row count`.
2. **Query 1:** fraud rate, fraud count and fraud value by `payment_method` and `device_type`, sorted by fraud value. Include a purpose comment.
3. **Query 2 (window function):** for each user with at least 2 transactions, the time gap in minutes between consecutive transactions (`LAG`), then the number of users who had a gap under 5 minutes. Compare the result with the velocity flag.
4. **Query 3 (CTE):** the top 20 users by fraud value, with their transaction count, fraud count, first and last transaction date.
5. **Query 4:** list `shipping_city` values with at least 500 transactions and a fraud rate above the overall rate, using a subquery for the overall rate.
6. **Reconcile:** for queries 1 and 4, recompute the same figures in pandas and assert the results match (tolerance 0.01 for money, 0 for counts). Print both.

### Acceptance checks
- Every query has a purpose comment. Pandas and SQL match.
- SQL files can be copied out and run as written (also save them in `fraud_queries.sql`).

In [25]:
# ---- F1. Load to SQLite ------------------------------------------------------------------------------------
# TODO: conn = sqlite3.connect("fraud_clean.db"); df_sql.to_sql("transactions", conn, if_exists="replace", index=False)

In [26]:
# ---- F2. Queries 1-4 and pandas reconciliation ----------------------------------------------------------------
q1_sql = """
-- Purpose: TODO
SELECT 1;
"""
# TODO: q2_sql, q3_sql, q4_sql, run each with pd.read_sql_query and reconcile against pandas.

In [27]:
# record("F", "SQL and pandas results reconcile for queries 1 and 4", <bool>, <evidence>)

---
## Part G. Segment fairness and drift

A model that works on average but fails a customer group creates false declines, complaints and regulatory risk.

### Requirements
1. For your chosen model and threshold, on the test set, compute **precision, recall and false-positive rate per segment** for: `payment_method`, `device_type`, `shipping_city` (cities with at least 300 test rows), and `data_source`. Show the **ratio of each segment FPR to overall FPR**.
2. Flag any segment whose FPR is more than 1.5 times the overall rate, or whose recall is more than 10 points below overall. Say whether the sample is big enough to conclude anything.
3. **Drift:** compute the **Population Stability Index (PSI)** for `amount_aed`, `ip_risk_score`, `card_age_days`, `transactions_last_1h` between train and test, and compare the fraud rate by month. Rule of thumb: PSI < 0.1 stable, 0.1 to 0.25 moderate, above 0.25 high. Write the PSI function yourself and test it on identical samples (it must return about 0).
4. **Monitoring plan:** list 5 metrics you would put on a dashboard in production, with an alert rule for each (for example "weekly PR-AUC drops by more than X", "PSI above 0.25 on any top-5 feature"). Keep it to a table.

### Acceptance checks
- PSI function has a self-test. Small segments are not over-interpreted.

In [28]:
# ---- G1-G2. Segment metrics --------------------------------------------------------------------------------------
# TODO

In [29]:
# ---- G3. PSI function (with self-test) and drift table -----------------------------------------------------------
# TODO: def psi(expected, actual, bins=10): ...
# TODO: assert psi(x, x) < 1e-6   # sanity self-test

In [30]:
# record("G", "PSI self-test passes; segment FPR ratios reported", <bool>, <evidence>)

---
## Part H. Manager memo and self-review

### H1. Memo to the VP of Risk (maximum 250 words)
Structure: **3 findings with numbers, 1 risk, 3 recommendations, 1 open question.**
- Use **only numbers printed in this notebook.** Count the words with code and print the count.
- Cover: where fraud is concentrated, how the rule engine performs, model vs rules at the chosen operating point, estimated saving with the stated assumptions, and the leakage caveat on the rule flags.
- Write for a non-technical reader. No jargon without a one-line explanation.

### H2. Self-review (automatic)
Run the cell below. Every Part A to G must have a PASS or FAIL line with evidence. A FAIL you reported honestly is better than a PASS you cannot prove.

### H3. Submission note (markdown cell at the end)
Dataset name and row count, the data issues you found and how you handled each, the assumptions you made (especially Part E), what you did **not** do and why, and the one thing you would investigate next with more data.

In [31]:
# ---- H1. Memo (write as a string, then count words) ---------------------------------------------------------------
memo = """
TODO: write the memo here using only printed numbers.
"""
# print("Word count:", len(memo.split()))

In [32]:
# ---- H2. Self-review table ------------------------------------------------------------------------------------------
review = pd.DataFrame(SELF_REVIEW)
display(review if len(review) else pd.DataFrame({"note": ["No Part has been recorded yet"]}))

,note
0,No Part has been recorded yet


---
## Scoring rubric (100 points)
| Area | Points | What earns full marks |
|---|---:|---|
| A. Data audit | 15 | Real counts only, every decision justified, row bridge balances, derived columns re-derived and tested |
| B. EDA | 10 | Reconciled breakdowns, intervals, low-volume groups marked, three insights with business "so what" |
| C. Rule engine | 15 | Precision/recall from a printed confusion matrix, flags re-derived, `is_fraud` vs flags relationship tested and explained |
| D. Modelling | 25 | Written leakage review, chronological split, two feature sets compared, test set used once, calibration, honest conclusion |
| E. Cost policy | 10 | Printed assumptions, cost curve, three-zone policy, sensitivity, saving vs rules |
| F. SQL | 5 | Queries correct and commented, pandas matches SQL |
| G. Fairness and drift | 5 | Segment FPR ratios, PSI with self-test, monitoring table |
| H. Memo and notebook hygiene | 15 | Memo under 250 words with traceable numbers, runs top to bottom with zero errors, no hardcoded asserts, self-review completed |

**Automatic deductions:** hand-typed results (-5 each, max -20), a fitted step that touches test data (-10), accuracy used as the headline metric (-5), unsupported claim about the data (-3 each), a notebook that does not run end to end (-15).

## Before you submit
1. Restart the kernel and **Run All**. Zero errors, all outputs visible.
2. Every number in the memo appears in a printed output.
3. You can explain each Part in one sentence. In the review, I will ask why you chose your threshold, and what the rule flags are doing in your model.
4. Send the executed `.ipynb`, `fraud_queries.sql` and the charts. I will score with the rubric above.